# Data Scraping

This section documents the raw data collection used in the project. The goal is to build a reproducible data foundation for forecasting DK1 day-ahead electricity prices and for evaluating consumer flexibility under high-price or high-load conditions.

The model relies on three types of information: electricity prices, electricity consumption, and weather. Prices define the forecasting target, consumption is used later as a proxy for grid stress, and weather provides explanatory variables for renewable generation and demand patterns. All raw files are fetched automatically from public APIs and saved locally before any modelling or feature engineering is performed.

---
## Data sources

The scraping pipeline collects four raw datasets. Each dataset has a specific role in the final forecasting and decision-support setup.

| Source | API | Role in project |
|--------|-----|-----------------|
| Day-ahead electricity prices | Energi Data Service | Forecasting target: hourly DK1 prices |
| Electricity consumption | Energi Data Service | Proxy for system load and grid stress |
| Historical weather observations | Open-Meteo Archive API | Weather actuals used for model features and forecast simulation |
| Previous-run weather forecasts | Open-Meteo Previous Runs API | Real forecast errors used to calibrate synthetic forecasts |

### 1. Day-ahead electricity prices
The price data is fetched from **Energi Data Service**. Two overlapping datasets are used: `Elspotprices` for older history and `DayAheadPrices` for more recent observations. They are normalised to the same schema, merged, and deduplicated into one hourly price file.

The project focuses on DK1, the western Danish bidding area. DK1 prices are strongly affected by wind power availability, cross-border interconnector flows, and hourly demand patterns. This makes the series relevant for a resilience-oriented forecasting task: prices contain information about both market conditions and periods where flexibility may be valuable.

### 2. Hourly electricity consumption
Consumption data is also collected from **Energi Data Service**. The raw records are reported at grid-area level and then aggregated to DK1. This dataset is not the model target, but it is important for the evaluation part of the project. High consumption hours are treated as a proxy for peak-load periods, which allows the later simulation to assess whether recommendations could shift flexible demand away from stressed hours.

### 3. Weather actuals
Historical weather observations are fetched from the **Open-Meteo Archive API** for a representative DK1 location, using latitude 56.15 and longitude 8.45. Eight hourly weather variables are collected because Danish power prices are closely linked to weather-driven renewable generation and demand.

| Variable | Why it matters |
|----------|----------------|
| Wind speed and direction at 10 m and 100 m | Wind turbine production proxy; 100 m is closer to actual hub height |
| Shortwave radiation | Solar PV production proxy |
| Cloud cover | Affects how much solar radiation reaches panels |
| Temperature at 2 m | Drives heating and cooling demand |
| Mean sea level pressure | Captures large-scale weather regime shifts |

These weather actuals serve two purposes. First, they are used as the observed weather values against which historical forecasts can be evaluated. Second, they provide the base weather time series used when generating synthetic forecast features for the full 2021-2026 modelling period.

### 4. NWP weather forecasts
Numerical weather prediction (NWP) forecasts are fetched from the **Open-Meteo Previous Runs API**. Unlike weather actuals, these are *as-issued* forecasts: they show what the ECMWF forecast looked like before the target hour occurred. For each target hour, the API provides the forecast as it looked 1, 2, 3, 4, and 5 days ahead, stored as `_previous_day1` through `_previous_day5` columns.

This distinction matters because the price model should use information that would have been available at prediction time. The limitation is coverage: previous-run forecasts are only available from 2025 onwards. Therefore, the real forecast data is used mainly to estimate forecast error distributions by horizon. Those distributions are then used in the processing step to simulate realistic forecast features for earlier years.

---
## Running the data collection

The raw data collection is wrapped in a single function, `fetch_all()`, so the same data pull can be repeated without manually running each scraper. The function collects prices, consumption, weather actuals, and previous-run weather forecasts in sequence, then writes the raw outputs to the `data/` folder.

The date range below covers the full project period. The previous-run weather forecast query is automatically restricted to 2025 onwards because the API does not provide earlier forecast archives.

In [ ]:
from src.data.data_collection import fetch_all

results = fetch_all(start="2021-01-01", end="2026-04-28", price_area="DK1")

After the call completes, the `data/` folder contains the following raw input files. These files are intentionally kept close to the API outputs; the heavier transformations are handled in the next processing step.

| File | Content | Coverage |
|------|---------|----------|
| `weather_actuals_raw.csv` | Hourly weather observations for DK1 West | 2021-present |
| `weather_forecasts_raw.csv` | NWP previous-run forecasts for DK1 West | 2025-present |
| `consumption_dk1_raw.csv` | Hourly DK1 electricity consumption | 2021-present |
| `day_ahead_prices_dk1_raw.csv` | Hourly DK1 day-ahead electricity prices | 2021-present |

---
## Next step: Data processing

The scraped files are the raw data foundation, but they are not yet the final modelling table. The next step is handled by `src/data/data_processing.py`, which connects the weather actuals and forecast archives into model-ready forecast features.

The processing step has two main outputs:

| Processing output | How it is created | Why it is needed |
|-------------------|-------------------|------------------|
| `weather_error_distributions.csv` | Real previous-run forecasts are compared with weather actuals at 24, 48, 72, 96, and 120 hour horizons | Measures typical NWP bias and uncertainty by weather variable and lead time |
| `forecast_dataset.parquet` | Synthetic 120-hour weather forecasts are generated for each 12-hour issue time from 2021 onwards | Provides the weather forecast features used by the XGBoost price model |

This design avoids using future information. The model is trained on forecast-like weather inputs rather than perfect historical weather observations, making the later price forecasts closer to the information that would be available in a real decision-support setting.